# AIRMAN Aeronautics — Data Science Assessment
## Skynet + TOGA Intelligence | Data Scientist Intern
**Author:** Kaushik Gaur  
**Date:** 2026-05-15  
**Products:** Skynet + TOGA

---

This notebook performs a complete analysis of AIRMAN's aviation training data across 8 tasks:
1. Data Cleaning & Validation
2. Skynet Operations Analytics
3. Training Progress Analytics
4. TOGA Study Intelligence
5. Finance & Operational Risk
6. Explainable Cadet Risk Score
7. Visualizations / Dashboard
8. Executive Insight Summary

All reports are saved to `reports/`, all charts to `charts/`, and computed outputs to `data/`.


In [ ]:
# ── SETUP ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global dark chart style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#3a3d4d',
    'axes.labelcolor':  '#e0e0e0',
    'axes.titlecolor':  '#ffffff',
    'xtick.color':      '#b0b0b0',
    'ytick.color':      '#b0b0b0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2a2d3d',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
})
PALETTE = ['#4e9af1','#f1c40f','#2ecc71','#e74c3c','#9b59b6','#1abc9c','#e67e22']
REF_DATE = pd.Timestamp('2026-05-15')

print("Setup complete ✅")


## Task 1 — Load & Validate Data

In [ ]:
# Load all datasets
sorties     = pd.read_csv('../data/sorties.csv')
aircraft    = pd.read_csv('../data/aircraft.csv')
cadets      = pd.read_csv('../data/cadets.csv')
instructors = pd.read_csv('../data/instructors.csv')
toga        = pd.read_csv('../data/toga_study.csv')
payments    = pd.read_csv('../data/payments.csv')

# Parse date columns
for col in ['scheduled_start','scheduled_end','actual_start','actual_end']:
    sorties[col] = pd.to_datetime(sorties[col], errors='coerce')

cadets['enrollment_date']     = pd.to_datetime(cadets['enrollment_date'])
payments['last_payment_date'] = pd.to_datetime(payments['last_payment_date'])
toga['last_active_date']      = pd.to_datetime(toga['last_active_date'])

print(f"Sorties:     {len(sorties)} rows")
print(f"Aircraft:    {len(aircraft)} rows")
print(f"Cadets:      {len(cadets)} rows")
print(f"Instructors: {len(instructors)} rows")
print(f"TOGA study:  {len(toga)} rows")
print(f"Payments:    {len(payments)} rows")


In [ ]:
# ── DATA QUALITY CHECKS ──────────────────────────────────────────────────────
issues = []

# 1. Missing values
for df, name in [(sorties,'sorties'),(aircraft,'aircraft'),(cadets,'cadets'),
                 (instructors,'instructors'),(toga,'toga'),(payments,'payments')]:
    mv = df.isnull().sum()
    mv = mv[mv > 0]
    for col, cnt in mv.items():
        issues.append(f"[MISSING] {name}.{col}: {cnt} missing values")

# 2. Completed sorties without actual start/end
mask = (sorties['status']=='completed') & (sorties['actual_start'].isna())
for idx in sorties[mask].index:
    issues.append(f"[DATA ERROR] Sortie {sorties.loc[idx,'sortie_id']}: completed but missing actual_start")

# 3. Delay mismatch validation
completed = sorties[sorties['status']=='completed'].copy()
completed['calc_delay'] = (
    (completed['actual_start'] - completed['scheduled_start']).dt.total_seconds() / 60
).round()
mismatch = completed[abs(completed['calc_delay'] - completed['delay_minutes']) > 2]
if len(mismatch) > 0:
    issues.append(f"[DELAY MISMATCH] {len(mismatch)} sorties have delay_minutes inconsistent with timestamps")

# 4. Payment arithmetic check
payments['calc_outstanding'] = payments['invoiced_amount'] - payments['paid_amount']
pay_err = payments[abs(payments['calc_outstanding'] - payments['outstanding_amount']) > 0]
if len(pay_err) > 0:
    issues.append(f"[PAYMENT ERROR] {len(pay_err)} payment records have incorrect outstanding_amount")

# 5. High defect count flag
hi_def = aircraft[aircraft['defect_count'] > 5]
for _, row in hi_def.iterrows():
    issues.append(f"[FLAG] Aircraft {row['aircraft_id']} ({row['registration']}): high defect_count = {row['defect_count']}")

# 6. Duplicate ID check
for df, col, name in [(sorties,'sortie_id','sorties'),(cadets,'cadet_id','cadets')]:
    dups = df[df.duplicated(col)]
    if not dups.empty:
        issues.append(f"[DUPLICATE] {name}.{col}: {len(dups)} duplicates found")

print(f"Total issues found: {len(issues)}")
for i in issues:
    print(f"  • {i}")


## Task 2 — Skynet Operations Analytics

In [ ]:
# ── AIRCRAFT UTILIZATION ─────────────────────────────────────────────────────
# Compute actual flight hours from sorties timestamps
completed['actual_duration_h'] = (
    (completed['actual_end'] - completed['actual_start']).dt.total_seconds() / 3600
)
ac_hours = completed.groupby('aircraft_id')['actual_duration_h'].sum().reset_index()
ac_hours.columns = ['aircraft_id','actual_flown_hours']

ac_util = aircraft.merge(ac_hours, on='aircraft_id', how='left').fillna(0)
ac_util['utilization_pct'] = (ac_util['actual_flown_hours'] / ac_util['total_available_hours'] * 100).round(1)
ac_util['downtime_pct']    = (ac_util['maintenance_downtime_hours'] / ac_util['total_available_hours'] * 100).round(1)

display(ac_util[['registration','type','total_available_hours','maintenance_downtime_hours',
                  'actual_flown_hours','utilization_pct','defect_count']])


In [ ]:
# ── INSTRUCTOR UTILIZATION ───────────────────────────────────────────────────
inst_sorties = completed.groupby('instructor_id').agg(
    sorties_done=('sortie_id','count'),
    flight_h_logged=('actual_duration_h','sum')
).reset_index()

inst_util = instructors.merge(inst_sorties, on='instructor_id', how='left').fillna(0)
inst_util['flight_to_duty_ratio'] = (
    inst_util['total_flight_hours'] / inst_util['total_duty_hours'] * 100
).round(1)

display(inst_util[['name','base_id','total_duty_hours','total_flight_hours',
                    'flight_to_duty_ratio','sorties_done']])


In [ ]:
# ── DISPATCH RELIABILITY ─────────────────────────────────────────────────────
total      = len(sorties)
comp_n     = (sorties['status']=='completed').sum()
cancel_n   = (sorties['status']=='cancelled').sum()
delayed_n  = ((sorties['status']=='completed') & (sorties['delay_minutes'] > 0)).sum()
avg_delay  = sorties[sorties['status']=='completed']['delay_minutes'].mean()

print(f"Total sorties:      {total}")
print(f"Completed:          {comp_n} ({comp_n/total*100:.1f}%)")
print(f"Cancelled:          {cancel_n} ({cancel_n/total*100:.1f}%)")
print(f"Completed w/ delay: {delayed_n}")
print(f"Average delay:      {avg_delay:.1f} min")
print()
print("Cancellation reasons:")
print(sorties[sorties['status']=='cancelled']['cancel_reason'].value_counts())


## Task 3 — Training Progress Analytics

In [ ]:
# Cadet progress metrics
cadets['progress_pct']     = (cadets['total_flown_hours'] / cadets['total_required_hours'] * 100).round(1)
cadets['remaining_hours']  = cadets['total_required_hours'] - cadets['total_flown_hours']
cadets['days_enrolled']    = (REF_DATE - cadets['enrollment_date']).dt.days
cadets['flying_rate']      = (cadets['total_flown_hours'] / cadets['days_enrolled']).round(3)
cadets['est_days_left']    = (cadets['remaining_hours'] / cadets['flying_rate']).round(0)
cadets['completion_risk']  = cadets['progress_pct'].apply(
    lambda p: 'Low' if p>=60 else ('Medium' if p>=30 else 'High'))

display(cadets[['cadet_id','name','course','progress_pct','remaining_hours',
                 'flying_rate','est_days_left','completion_risk']])


## Task 4 — TOGA Study Intelligence

In [ ]:
# Subject-level analysis
toga['chapter_pct']      = (toga['chapters_completed'] / toga['total_chapters'] * 100).round(1)
toga['days_inactive']    = (REF_DATE - toga['last_active_date']).dt.days
toga['practice_factor']  = toga['practice_tests_attempted'].apply(lambda x: min(x*10, 40))
toga['subject_readiness']= (
    toga['chapter_pct'] * 0.4 +
    toga['avg_quiz_score'] * 0.4 +
    toga['practice_factor'] * 0.2
).round(1)

display(toga[['cadet_id','subject','chapter_pct','avg_quiz_score',
              'days_inactive','subject_readiness']])


## Task 5 — Finance & Operational Risk

In [ ]:
# Payment metrics
payments['payment_pct']     = (payments['paid_amount'] / payments['invoiced_amount'] * 100).round(1)
payments['outstanding_pct'] = (payments['outstanding_amount'] / payments['invoiced_amount'] * 100).round(1)
payments['days_since_pay']  = (REF_DATE - payments['last_payment_date']).dt.days

pay = payments.merge(cadets[['cadet_id','name','course']], on='cadet_id')

def payment_risk_score(row):
    score = row['outstanding_pct'] * 0.5
    score += 30 if row['days_since_pay'] > 30 else (15 if row['days_since_pay'] > 15 else 5)
    score += min(row['outstanding_amount'] / 5000, 20)
    return min(round(score, 1), 100)

pay['pay_risk_score'] = pay.apply(payment_risk_score, axis=1)
pay['pay_risk_level'] = pay['pay_risk_score'].apply(
    lambda s: 'High' if s>=60 else ('Medium' if s>=30 else 'Low'))

display(pay[['name','course','invoiced_amount','outstanding_amount',
             'outstanding_pct','days_since_pay','pay_risk_score','pay_risk_level']])


## Task 6 — Explainable Cadet Risk Score

In [ ]:
# ── RISK SCORE FORMULA ───────────────────────────────────────────────────────
# Transparent weighted additive model — no black-box ML

def norm(val, lo, hi, invert=True):
    """Normalize val to [0,1]. invert=True means higher val = lower risk."""
    if invert:
        return max(0, min(1, (hi - val) / (hi - lo)))
    else:
        return max(0, min(1, (val - lo) / (hi - lo)))

def compute_risk(row):
    return round(
        norm(row['progress_pct'], 0, 100, invert=True)     * 25 +  # low progress = high risk
        norm(row['avg_quiz'],     0,  80, invert=True)     * 15 +  # low quiz = high risk
        norm(row['study_pct'],    0, 100, invert=True)     * 15 +  # low study = high risk
        norm(row['days_inactive'],0,  30, invert=False)    * 10 +  # high inactivity = high risk
        norm(row['outstanding_pct'],0,100, invert=False)   * 20 +  # high payment = high risk
        norm(row['cancel_rate'], 0,  80, invert=False)     * 10 +  # high cancel = high risk
        norm(row['avg_delay'],   0,  60, invert=False)     *  5,   # high delay = high risk
    1)

# Build feature table
cadet_sorties = sorties.groupby('cadet_id').agg(
    cancel_rate=('status', lambda x: (x=='cancelled').sum()/len(x)*100),
    avg_delay=('delay_minutes','mean')
).reset_index()

study_agg = toga.groupby('cadet_id').agg(
    avg_quiz=('avg_quiz_score','mean'),
    study_pct=('chapter_pct','mean'),
    days_inactive=('last_active_date', lambda x: (REF_DATE - x.max()).days)
).reset_index()

risk_df = cadets[['cadet_id','name','course','progress_pct']].copy()
risk_df = risk_df.merge(cadet_sorties, on='cadet_id', how='left')
risk_df = risk_df.merge(study_agg, on='cadet_id', how='left')
risk_df = risk_df.merge(payments[['cadet_id','outstanding_pct']], on='cadet_id', how='left')
risk_df = risk_df.fillna({'avg_quiz':0,'study_pct':0,'days_inactive':30,'cancel_rate':0,'avg_delay':0})

risk_df['risk_score'] = risk_df.apply(compute_risk, axis=1)
risk_df['risk_level'] = risk_df['risk_score'].apply(
    lambda s: 'Low' if s<40 else ('Medium' if s<70 else 'High'))

display(risk_df[['cadet_id','name','course','risk_score','risk_level']])


## Task 7 — Visualizations

All 7 charts are pre-generated in `charts/`. Run the cells below to regenerate or display them.

In [ ]:
# Display all charts inline
from IPython.display import Image, display as ipy_display
import os

chart_files = [
    ('charts/aircraft_utilization.png',    'Chart 1: Aircraft Utilization'),
    ('charts/cancellation_reasons.png',    'Chart 2: Cancellation Reasons'),
    ('charts/cadet_progress.png',          'Chart 3: Cadet Flight Progress'),
    ('charts/study_readiness.png',         'Chart 4: TOGA Study Readiness'),
    ('charts/payment_risk.png',            'Chart 5: Payment Risk'),
    ('charts/cadet_risk_scores.png',       'Chart 6: Cadet Risk Scores'),
    ('charts/flight_vs_study_progress.png','Chart 7: Flight vs Study Progress'),
]

for path, title in chart_files:
    if os.path.exists(f'../{path}'):
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
        ipy_display(Image(filename=f'../{path}', width=900))
    else:
        print(f"Chart not found: {path}")


## Summary — Key Findings

| Cadet | Flight Progress | Study Readiness | Payment Risk | Risk Score | Level |
|-------|----------------|-----------------|--------------|------------|-------|
| C001 Arjun Menon | 63.3% ✅ | 59.6 🟡 | Low ✅ | 22.5 | Low |
| C002 Meera Iyer | 37.0% 🟡 | 37.2 🔴 | Medium 🟡 | 45.9 | Medium |
| C003 Rahul Nair | 26.7% 🔴 | 25.5 🔴 | High 🔴 | 62.8 | Medium |

See `reports/executive_insights.md` for full leadership report.